# 讓畫面知道 Agent 跑到哪一步這份教材的主題不是完整追蹤系統，而是 Chat WebUI 在產生最終文字前，要怎麼顯示「現在正在哪個工作流階段」。SDK 會把階段事件交給 `event_callback`，WebUI 只要讀事件裡的 `label` 就能更新畫面。

## 在 Colab 準備環境如果你是在 Colab 開啟，先複製專案並切換到專案資料夾；如果你已經在專案資料夾，可以直接跳過這格。

In [ ]:
!git clone https://github.com/R300-AI/Agentic-SDK.git%cd Agentic-SDK

## 載入流程元件這次仍然用最小問答流程。重點不是重新解釋每個模組，而是看 `Workflow` 如何對外送出階段事件。

In [ ]:
from agentic_sdk import Workflowfrom agentic_sdk.modules import DirectAnswerAction, KeywordRetrieve, PassThroughPerceive

## 先設定畫面要顯示的階段文字`stage_labels` 是 `Workflow` 的設定。Chat WebUI 不需要自己猜每個模組要顯示什麼，只要使用 SDK 事件裡的 `label`。這裡只覆寫本章想示範的文字；沒有設定的階段會使用 SDK 預設值。

In [ ]:
stage_labels = {    'perceive': '正在理解你的問題',    'retrieve': '正在查找參考資料',    'action': '正在準備回覆',}stage_labels

## 建立一條會送出階段事件的流程把 `stage_labels` 放進 `Workflow`。之後流程每跑到一個模組，事件裡就會帶出對應的 `stage` 和 `label`。

In [ ]:
workflow = Workflow(    workflow_name='WebUI 階段提示 Agent',    stage_labels=stage_labels,    perceive=PassThroughPerceive(),    retrieve=KeywordRetrieve(items=[{'keywords': ['sdk'], 'content': 'Agentic SDK 可以把工作流階段交給畫面顯示。'}]),    action=DirectAnswerAction(),)

## 準備最單純的事件接收函式MVP 先不要在 notebook 裡重組事件。`on_event` 直接把 SDK 送出的事件格式印出來，讓你看到 Chat WebUI 實際會收到什麼。

In [ ]:
def on_event(event):    print(event)

## 執行時把事件接收函式傳進去這個寫法就是 Chat WebUI 可以接的最小模式：`Workflow` 執行時把階段事件丟給 `event_callback`，畫面收到事件後顯示 `event["label"]`。

In [ ]:
user_message = '請介紹 SDK 的階段提示方式'result = workflow.run(    user_message,    event_callback=on_event,)

## 最後再看主體回覆階段事件會先印出來；等流程完成後，才讀取最後要顯示給使用者的文字。

In [ ]:
print(result.final_message)

## Chat WebUI 要怎麼接正式畫面不需要印出整包事件。收到 `type == "stage"` 且 `status == "running"` 的事件時，更新同一個狀態列即可；等最後文字開始顯示，就把狀態列收起來或改成「正在生成回覆」。

In [ ]:
def webui_on_event(event):    if event.get('type') == 'stage' and event.get('status') == 'running':        print('畫面狀態：' + event.get('label', '正在處理'))